# Full Baseline Pipeline

## 1. Install

In [1]:
!pip install -q timm albumentations

## 2. Imports

In [2]:
import os
import cv2
import timm
import torch
import random
import numpy as np
import pandas as pd

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm.auto import tqdm

## 3. Config

In [3]:
class CFG:

    img_size = 256
    batch_size = 32
    epochs = 12

    lr = 3e-4
    weight_decay = 1e-4

    n_folds = 5
    num_classes = 8

    device = "cuda"

    image_dir = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/images"

    train_csv = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/train.csv"
    add_csv = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/additional_train.csv"
    test_csv = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/test.csv"

## 4A. Load train dataset

In [4]:
train = pd.read_csv(CFG.train_csv)
add = pd.read_csv(CFG.add_csv)

df = pd.concat([train, add], ignore_index=True)

# remove unlabeled samples if present
df = df[df.pen_id != -1].reset_index(drop=True)

print("Training samples:", len(df))
print(df.head())

Training samples: 40250
   image_id        image_path writer_id  pen_id
0         4  images/00004.png       W27       8
1         5  images/00005.png       W17       1
2         7  images/00007.png       W01       8
3         8  images/00008.png       W17       5
4         9  images/00009.png       W24       4


## 4B. Load test dataset

In [5]:
test_df = pd.read_csv(CFG.test_csv)

print("Test samples:", len(test_df))
print(test_df.head())

Test samples: 5905
                              image_id  \
0  v2_8eb750cb7bac5c42036af72b8253976b   
1  v2_04e19b0acea03fe2ae474ce8a4c6b705   
2  v2_6b3400d5252124adcc9859cbc78c5d8a   
3  v2_79025cb2ef36af3dc15c91056fa225dc   
4  v2_3ed2f62c32c69ec191b7e5b86433cb87   

                                       image_path  
0  images/v2_8eb750cb7bac5c42036af72b8253976b.png  
1  images/v2_04e19b0acea03fe2ae474ce8a4c6b705.png  
2  images/v2_6b3400d5252124adcc9859cbc78c5d8a.png  
3  images/v2_79025cb2ef36af3dc15c91056fa225dc.png  
4  images/v2_3ed2f62c32c69ec191b7e5b86433cb87.png  


## 4C. Assign full image_path

In [6]:
df["image_path"] = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/" + df["image_path"]
test_df["image_path"] = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/" + test_df["image_path"]

## 4D. debug 1

In [7]:
df["pen_id"] = df["pen_id"] - 1

## 5A. Sharpness-Focused Augmentation

In [8]:
train_tfms = A.Compose([

    A.RandomResizedCrop(
        size=(CFG.img_size, CFG.img_size),
        scale=(0.80, 1.0),
        ratio=(0.95, 1.05),
        p=1.0
    ),

    A.HorizontalFlip(p=0.5),

    # emphasize stroke sharpness
    A.Sharpen(alpha=(0.2, 0.6), lightness=(0.8, 1.2), p=0.5),

    # subtle blur for robustness
    A.GaussianBlur(blur_limit=(3,5), p=0.2),

    # micro-noise helps generalization
    A.GaussNoise(var_limit=(5, 15), p=0.4),

])
valid_tfms = A.Compose([

    A.Resize(
        height=CFG.img_size,
        width=CFG.img_size
    )

])

/tmp/ipykernel_55/3014366468.py:19: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 15), p=0.4),


## 5B. Ring Crop (remove useless center)

In [9]:
def ring_crop(img):

    h,w,_ = img.shape
    cx,cy = w//2, h//2

    r_outer = int(min(cx,cy)*0.95)
    r_inner = int(min(cx,cy)*0.35)

    Y,X = np.ogrid[:h,:w]

    dist = (X-cx)**2 + (Y-cy)**2

    mask = (dist <= r_outer**2) & (dist >= r_inner**2)

    out = img.copy()
    out[~mask] = 0

    return out

## 5C. Laplacian Stroke Texture Channel

In [10]:
def laplacian_channel(img):

    gray = cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)

    lap = cv2.Laplacian(gray,cv2.CV_32F)

    lap = cv2.normalize(lap,None,0,255,cv2.NORM_MINMAX)

    return lap.astype(np.uint8)

## 5D. Ink Density Channel

In [11]:
def ink_density_channel(img):

    gray = cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)

    blur = cv2.GaussianBlur(gray,(5,5),0)

    density = cv2.subtract(gray,blur)

    density = cv2.normalize(density,None,0,255,cv2.NORM_MINMAX)

    return density.astype(np.uint8)

## 6. Dataset (train)

In [12]:
class CircleDataset(Dataset):

    def __init__(self, df, transforms=None):

        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # image_path is already full path
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # remove center
        img = ring_crop(img)

        # texture channels BEFORE augmentation
        lap = laplacian_channel(img)
        ink = ink_density_channel(img)

        # stack channels
        lap = lap[..., None]
        ink = ink[..., None]

        img = np.concatenate([img, lap, ink], axis=2)

        # apply augmentation
        if self.transforms:
            img = self.transforms(image=img)["image"]

        # normalize
        img = img.astype(np.float32) / 255.0

        img = torch.from_numpy(img).permute(2, 0, 1)

        label = row.pen_id

        return img, label

## 7. MixUp / CutMix

In [13]:
def mixup_data(x, y, alpha=0.4):

    lam = np.random.beta(alpha, alpha)

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda()

    mixed_x = lam * x + (1 - lam) * x[index]

    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

## 8. Model

In [14]:
class ResNet18Pen(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "resnet18",
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )

        # modify first conv to accept 5 channels
        old_conv = self.backbone.conv1

        self.backbone.conv1 = nn.Conv2d(
            5,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        # copy pretrained weights for RGB channels
        with torch.no_grad():
            self.backbone.conv1.weight[:, :3] = old_conv.weight
            self.backbone.conv1.weight[:, 3:] = old_conv.weight[:, :2]

        self.head = nn.Sequential(
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, CFG.num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

## 9. SWA Setup

In [15]:
from torch.optim.swa_utils import AveragedModel, SWALR

## 10. Training Loop

In [16]:
def train_fn(model, loader, optimizer, criterion):

    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc="Train", leave=False)

    for x, y in pbar:

        x, y = x.cuda(), y.cuda()

        if random.random() < 0.5:
            x, ya, yb, lam = mixup_data(x, y)

            pred = model(x)
            loss = lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

        else:
            pred = model(x)
            loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # show running loss
        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

## 11. Validation

In [17]:
def valid_fn(model, loader):

    model.eval()

    preds = []
    targets = []

    pbar = tqdm(loader, desc="Valid", leave=False)

    with torch.no_grad():
        for x, y in pbar:

            x = x.cuda()

            pred = model(x)

            preds.append(pred.softmax(1).cpu().numpy())
            targets.append(y.numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    acc = (preds.argmax(1) == targets).mean()

    return acc

## 12. KFold Training

In [18]:
print(df.pen_id.min(), df.pen_id.max())

0 7


In [ ]:
skf = StratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=42)

models = []

for fold,(tr_idx,val_idx) in enumerate(skf.split(df,df.pen_id)):

    train_df = df.iloc[tr_idx]
    val_df = df.iloc[val_idx]

    train_ds = CircleDataset(train_df,train_tfms)
    val_ds = CircleDataset(val_df,valid_tfms)

    train_loader = DataLoader(train_ds,batch_size=CFG.batch_size,shuffle=True)
    val_loader = DataLoader(val_ds,batch_size=CFG.batch_size)

    model = ResNet18Pen().cuda()

    optimizer = torch.optim.AdamW(model.parameters(),lr=CFG.lr)

    criterion = nn.CrossEntropyLoss()

    swa_model = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=3e-4, anneal_strategy="cos")

    for epoch in range(CFG.epochs):

        train_loss = train_fn(model,train_loader,optimizer,criterion)

        acc = valid_fn(model,val_loader)

        print(f"fold {fold} epoch {epoch} acc {acc}")

        if epoch > CFG.epochs*0.75:

            swa_model.update_parameters(model)
            swa_scheduler.step()

    torch.save(swa_model.state_dict(),f"model_fold{fold}.pth")

    models.append(swa_model)

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Train:   0%|          | 0/1007 [00:00<?, ?it/s]

Valid:   0%|          | 0/252 [00:00<?, ?it/s]

fold 0 epoch 0 acc 0.7320496894409938


Train:   0%|          | 0/1007 [00:00<?, ?it/s]

## 13. Test Dataset

In [ ]:
class TestDataset(Dataset):

    def __init__(self, df, transforms=None):

        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # image_path already contains full path
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # same preprocessing as training
        img = ring_crop(img)

        lap = laplacian_channel(img)
        ink = ink_density_channel(img)

        lap = lap[..., None]
        ink = ink[..., None]

        img = np.concatenate([img, lap, ink], axis=2)

        if self.transforms:
            img = self.transforms(image=img)["image"]

        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2,0,1)

        return img

## 14. TTA Inference

In [ ]:
def inference(models,loader):

    preds=[]

    for model in models:

        model.eval()

        fold_preds=[]

        with torch.no_grad():

            for x in loader:

                x=x.cuda()

                p=model(x).softmax(1)

                # horizontal flip TTA
                p2=model(torch.flip(x,[3])).softmax(1)

                p=(p+p2)/2

                fold_preds.append(p.cpu().numpy())

        preds.append(np.concatenate(fold_preds))

    return np.mean(preds,axis=0)

## 15. Submission

In [ ]:
test = pd.read_csv("/kaggle/input/competitions/icdar-2026-circleid-pen-classification/sample_submission.csv")

test_ds = TestDataset(test)

test_loader = DataLoader(test_ds, batch_size=32)

preds = inference(models, test_loader)

# convert 0..7 → 1..8
test["pen_id"] = preds.argmax(1) + 1

test[["image_id", "pen_id"]].to_csv("submission.csv", index=False)